# 4. Mask-only control

Describes each mask by eight shape descriptors (no image, no lesion position) and classifies them
with the same weighted ensemble in the same outer folds, for expert and out-of-fold predicted masks.
Run after notebooks 1-3.

In [ ]:
from pathlib import Path

import cv2
import joblib
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import f1_score, recall_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

DATASET = "BUSI"  # or "BUS-UCLM"
OUT_DIR = Path("../outputs") / DATASET
WEIGHTS = [1, 1, 2, 1]
SEED = 42

data = joblib.load(OUT_DIR / "folds.pkl")
y, folds = data["y"], data["folds"]
masks = joblib.load(OUT_DIR / "masks.pkl")

## Shape descriptors

Empty-mask indicator, area fraction, number of regions, and for the largest region: solidity, extent,
eccentricity, compactness (perimeter²/4π·area) and bounding-box aspect ratio. Empty masks get zeros.

In [ ]:
def describe(mask):
    m = (mask > 0).astype(np.uint8)
    if m.sum() == 0:
        return [1, 0, 0, 0, 0, 0, 0, 0]
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    c = max(contours, key=cv2.contourArea)
    area = max(cv2.contourArea(c), 1.0)
    hull = max(cv2.contourArea(cv2.convexHull(c)), 1.0)
    _, _, w, h = cv2.boundingRect(c)
    ecc = 0.0
    if len(c) >= 5:
        _, axes, _ = cv2.fitEllipse(c)
        major, minor = max(axes), max(min(axes), 1e-6)
        ecc = np.sqrt(1 - (minor / major) ** 2)
    return [0, m.mean(), len(contours), area / hull, area / (w * h), ecc,
            cv2.arcLength(c, True) ** 2 / (4 * np.pi * area), w / h]

## Ensemble on the outer folds

In [ ]:
def classifiers():
    return [DecisionTreeClassifier(random_state=0),
            KNeighborsClassifier(5),
            RandomForestClassifier(100, random_state=0, n_jobs=-1),
            HistGradientBoostingClassifier(max_iter=100, max_depth=3, max_bins=64,
                                           early_stopping=False, random_state=0)]


def predict(features_per_fold):
    pred = np.zeros(len(y), int)
    for (tr, te), F in zip(folds, features_per_fold):
        scaler = StandardScaler().fit(F[tr])
        prob = np.stack([c.fit(scaler.transform(F[tr]), y[tr]).predict_proba(scaler.transform(F[te]))
                         for c in classifiers()])
        pred[te] = np.tensordot(np.asarray(WEIGHTS) / sum(WEIGHTS), prob, 1).argmax(-1)
    return pred


def summarize(pred):
    rng = np.random.default_rng(SEED)
    boot = [f1_score(y[i], pred[i], average="macro")
            for i in (rng.integers(0, len(y), len(y)) for _ in range(1000))]
    fold_f1 = [f1_score(y[te], pred[te], average="macro") for _, te in folds]
    recall = recall_score(y, pred, average=None)
    return dict(macro_f1=f1_score(y, pred, average="macro"),
                ci_low=np.percentile(boot, 2.5), ci_high=np.percentile(boot, 97.5),
                fold_mean=np.mean(fold_f1), fold_sd=np.std(fold_f1, ddof=1),
                recall_benign=recall[0], recall_malignant=recall[1], recall_normal=recall[2])

## Run

In [ ]:
expert = np.array([describe(m) for m in masks["expert"]], float)
predicted = [np.array([describe(m) for m in fold_masks], float) for fold_masks in masks["predicted"]]

preds = {"expert": predict([expert] * len(folds)), "predicted": predict(predicted)}
mask_only = pd.DataFrame([dict(dataset=DATASET, masks=name, **summarize(p)) for name, p in preds.items()]).round(3)
mask_only.to_csv(OUT_DIR / "mask_only.csv", index=False)
mask_only

## Expert mask only versus oracle fusion (exact McNemar)

In [ ]:
runs = pd.DataFrame(joblib.load(OUT_DIR / "ga_runs.pkl"))
oracle = np.zeros(len(y), int)
for _, r in runs.query("config == 'Standard' and seed == 0 and condition == 'oracle'").iterrows():
    oracle[folds[r.fold][1]] = r.pred

a, b = preds["expert"] == y, oracle == y
only_a, only_b = int((a & ~b).sum()), int((~a & b).sum())
print(f"{DATASET}: expert mask only {only_a} / oracle {only_b}, exact p = {binomtest(only_a, only_a + only_b).pvalue:.2f}")